In [3]:
import pandas as pd
from pathlib import Path
import sys
from sqlalchemy import text

# Add the RetailIQ project root to Python's search path
PROJECT_DIR = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_DIR))

# Now Python can find database/db_connection.py
from database.db_connection import get_engine
# Load order, customer, and product information
engine = get_engine()

try:
    with engine.connect() as conn:
        sales = pd.read_sql(
            text("""
                SELECT
                    Order_ID,
                    Customer_ID,
                    Product_ID,
                    Order_Status
                FROM sales
                WHERE Order_Status = 'Delivered'
            """),
            conn
        )

        products = pd.read_sql(
            text("""
                SELECT Product_ID, Product_Name, Category
                FROM products
            """),
            conn
        )
finally:
    engine.dispose()

print("Delivered sales records:", len(sales))
print("Unique orders:", sales["Order_ID"].nunique())
print("Unique products purchased:", sales["Product_ID"].nunique())

display(sales.head())


Delivered sales records: 200139
Unique orders: 200139
Unique products purchased: 2000


,Order_ID,Customer_ID,Product_ID,Order_Status
0,ORD0000000001,CUST00014303,PROD001017,Delivered
1,ORD0000000002,CUST00034253,PROD000703,Delivered
2,ORD0000000004,CUST00026507,PROD000213,Delivered
3,ORD0000000005,CUST00009418,PROD001437,Delivered
4,ORD0000000008,CUST00017326,PROD001392,Delivered


In [4]:

# One row per unique product in each delivered order
order_products = sales[
    ["Order_ID", "Product_ID"]
].drop_duplicates()

# Count how often each product appears in delivered orders
product_frequency = (
    order_products.groupby("Product_ID")["Order_ID"]
    .nunique()
    .sort_values(ascending=False)
)

print("Unique order-product pairs:", len(order_products))
display(product_frequency.head(10))


Unique order-product pairs: 200139


Product_ID
PROD000176    219
PROD000914    215
PROD001615    215
PROD000949    214
PROD000530    214
PROD000074    212
PROD001297    212
PROD000246    211
PROD001929    210
PROD001824    209
Name: Order_ID, dtype: int64

In [5]:
# Create a product-to-product co-occurrence matrix
basket = (
    order_products
    .assign(purchased=1)
    .pivot_table(
        index="Order_ID",
        columns="Product_ID",
        values="purchased",
        fill_value=0
    )
)

# Count how often pairs of products occur in the same order
co_occurrence = basket.T.dot(basket)

print("Basket matrix shape:", basket.shape)
print("Co-occurrence matrix shape:", co_occurrence.shape)

display(co_occurrence.iloc[:5, :5])

Basket matrix shape: (200139, 2000)
Co-occurrence matrix shape: (2000, 2000)


Product_ID,PROD000001,PROD000002,PROD000003,PROD000004,PROD000005
Product_ID,,,,,
PROD000001,174.0,0.0,0.0,0.0,0.0
PROD000002,0.0,192.0,0.0,0.0,0.0
PROD000003,0.0,0.0,175.0,0.0,0.0
PROD000004,0.0,0.0,0.0,207.0,0.0
PROD000005,0.0,0.0,0.0,0.0,158.0


In [6]:
# Count distinct products purchased by each customer
customer_product_counts = (
    sales.groupby("Customer_ID")["Product_ID"]
    .nunique()
    .sort_values(ascending=False)
)

print("Customers with delivered purchases:", len(customer_product_counts))
print("\nProducts purchased per customer:")
print(customer_product_counts.describe())

print("\nCustomers with the most distinct products:")
display(customer_product_counts.head(10))

Customers with delivered purchases: 39726

Products purchased per customer:
count    39726.000000
mean         5.023486
std          2.191356
min          1.000000
25%          3.000000
50%          5.000000
75%          6.000000
max         17.000000
Name: Product_ID, dtype: float64

Customers with the most distinct products:


Customer_ID
CUST00005290    17
CUST00010900    16
CUST00015974    16
CUST00015328    15
CUST00021395    15
CUST00014709    15
CUST00028135    15
CUST00012539    15
CUST00029612    14
CUST00012699    14
Name: Product_ID, dtype: int64

In [7]:
# Create a customer-product purchase matrix
customer_product_matrix = pd.crosstab(
    sales["Customer_ID"],
    sales["Product_ID"]
)

# Convert purchase counts to binary values
customer_product_matrix = (
    customer_product_matrix > 0
).astype("int8")

print("Matrix shape:", customer_product_matrix.shape)
print("Total purchases represented:", customer_product_matrix.sum().sum())

display(customer_product_matrix.iloc[:5, :10])

Matrix shape: (39726, 2000)
Total purchases represented: 199563


Product_ID,PROD000001,PROD000002,PROD000003,PROD000004,PROD000005,PROD000006,PROD000007,PROD000008,PROD000009,PROD000010
Customer_ID,,,,,,,,,,
CUST00000001,0,0,0,0,0,0,0,0,0,0
CUST00000002,0,0,0,0,0,0,0,0,0,0
CUST00000004,0,0,0,0,0,0,0,0,0,0
CUST00000005,0,0,0,0,0,0,0,0,0,0
CUST00000006,0,0,0,0,0,0,0,1,0,0


In [11]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(
    n_neighbors=21,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

knn.fit(customer_product_matrix)

print("Updated similarity model trained successfully.")

Updated similarity model trained successfully.


In [20]:

def recommend_products(customer_id, n_recommendations=5):
    # 1. Validate customer ID
    if customer_id not in customer_product_matrix.index:
        raise ValueError(f"Customer ID not found: {customer_id}")

    if n_recommendations < 1:
        raise ValueError("n_recommendations must be at least 1")

    # 2. Get the target customer's purchase history
    customer_vector = customer_product_matrix.loc[[customer_id]]

    # 3. Find the nearest customers
    distances, indices = knn.kneighbors(customer_vector)

    # 4. Exclude the target customer
    similar_indices = indices[0][1:]
    similar_distances = distances[0][1:]

    similar_customers = customer_product_matrix.iloc[similar_indices]

    # 5. Calculate similarity weights
    similarities = 1 - similar_distances

    # 6. Score products based on purchases by similar customers
    product_scores = similar_customers.T.dot(similarities)

    # 7. Exclude products already purchased by the target customer
    purchased_products = customer_product_matrix.loc[customer_id]
    product_scores = product_scores[purchased_products == 0]

    # 8. Keep only products with positive recommendation scores
    product_scores = product_scores[product_scores > 0]

    # 9. Rank products by score
    top_products = product_scores.sort_values(
        ascending=False
    ).head(n_recommendations)

    # 10. Prepare the recommendation results
    recommendations = (
        top_products.rename("Recommendation_Score")
        .rename_axis("Product_ID")
        .reset_index()
        .merge(
            products_full,
            on="Product_ID",
            how="left",
            validate="one_to_one"
        )
    )

    # 11. Make product names distinguishable
    recommendations["Display_Name"] = (
        recommendations["Product_Name"].astype(str)
        + " ("
        + recommendations["Product_ID"].astype(str)
        + ")"
    )

    # 12. Return the most useful columns first
    output_columns = [
        "Product_ID",
        "Display_Name",
        "Product_Name",
        "Category",
        "Recommendation_Score",
        "Selling_Price",
        "Avg_Rating",
        "Stock_Quantity"
    ]

    output_columns = [
        col for col in output_columns
        if col in recommendations.columns
    ]

    return recommendations[output_columns]


In [21]:
test_customer_id = "CUST00000001"

display(
    recommend_products(
        test_customer_id,
        n_recommendations=10
    )
)

,Product_ID,Display_Name,Product_Name,Category,Recommendation_Score,Selling_Price,Avg_Rating,Stock_Quantity
0,PROD000242,Fortune Oil V10 (PROD000242),Fortune Oil V10,Grocery,0.447214,212.37,4.51,172
1,PROD000225,Urban Ladder Sofa V1 (PROD000225),Urban Ladder Sofa V1,Home,0.447214,2667.19,4.33,572
2,PROD001259,Nescafe Coffee V9 (PROD001259),Nescafe Coffee V9,Grocery,0.447214,1411.27,4.35,101
3,PROD001293,Lays Chips V2 (PROD001293),Lays Chips V2,Grocery,0.447214,862.35,4.34,179
4,PROD000120,Urban Ladder Sofa V1 (PROD000120),Urban Ladder Sofa V1,Home,0.316228,8960.22,4.41,468
5,PROD000213,boAt Headphones V2 (PROD000213),boAt Headphones V2,Electronics,0.316228,34081.56,4.43,743
6,PROD000685,Plum Serum V4 (PROD000685),Plum Serum V4,Beauty,0.316228,449.15,4.38,462
7,PROD000856,Lakme Lipstick V9 (PROD000856),Lakme Lipstick V9,Beauty,0.316228,1572.82,4.44,958
8,PROD001173,Nivea Cream V10 (PROD001173),Nivea Cream V10,Beauty,0.316228,1381.82,4.35,551
9,PROD001202,Prestige Cooker V3 (PROD001202),Prestige Cooker V3,Home,0.316228,28296.47,4.48,499


In [22]:
import joblib
from pathlib import Path

# Locate the project's models folder
PROJECT_DIR = Path.cwd().resolve().parent
MODELS_DIR = PROJECT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Save the customer-product matrix and similarity model
joblib.dump(
    customer_product_matrix,
    MODELS_DIR / "customer_product_matrix.pkl"
)

joblib.dump(
    knn,
    MODELS_DIR / "product_recommender_knn.pkl"
)

# Save product details needed by the recommendation function
products_full.to_pickle(
    MODELS_DIR / "recommendation_products.pkl"
)

print("Product recommendation artifacts saved successfully.")

print("Saved files:")
for file in [
    "customer_product_matrix.pkl",
    "product_recommender_knn.pkl",
    "recommendation_products.pkl"
]:
    print(MODELS_DIR / file)

Product recommendation artifacts saved successfully.
Saved files:
D:\projects\RetailIQ\models\customer_product_matrix.pkl
D:\projects\RetailIQ\models\product_recommender_knn.pkl
D:\projects\RetailIQ\models\recommendation_products.pkl
